In [ ]:
!pip install enoslib ipywidgets

In [1]:
!ssh rennes.grid5000.fr hostname

frennes


### Building and pushing the docker images to dockerhub

In [6]:
!/home/corentin/fcquic_applications_master_thesis/docker_images/build_images.sh

Building base image

[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.47kB                                     0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.2s
[+] Building 0.3s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.47kB                                     0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
[+] Building 0.3s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.47kB                                     0.0s
 => [internal] load

In [7]:
!/home/corentin/fcquic_applications_master_thesis/docker_images/upload_images.sh

tagging and pushing base
The push refers to repository [docker.io/corentindetry/base]

596775ab: Waiting 
05cab8af: Waiting 
1c60fe54: Waiting 
17e0a25e: Waiting 
afebaf4d: Waiting 
12bd67ef: Waiting 
1b5e9776: Waiting 
b238bc9a: Waiting 
fca4d5c8: Waiting 
53f2dec3: Waiting 
afebaf4d: Pushed   28.24MB/28.24MBlatest: digest: sha256:0308cb2372889b6fd56fbf9c38f15d6f26c9dc699be822f089a85bc85e56e8d7 size: 856
tagging and pushing fcquic_client
The push refers to repository [docker.io/corentindetry/fcquic_client]

fe8e9fb7: Waiting 
12bd67ef: Waiting 
32e7cfef: Waiting 
fca4d5c8: Waiting 
53f2dec3: Waiting 
17e0a25e: Waiting 
1b5e9776: Waiting 
be19fd1f: Waiting 
890ea33a: Waiting 
afebaf4d: Waiting 
1c60fe54: Waiting 
b238bc9a: Waiting 
2e7cfef: Pushed   1.287kB/1.287kBry/base Klatest: digest: sha256:f4d846e0c94c27c0a79b49f3469fc7dd1cd475a9bb66be3b5beabbfad83ec181 size: 856
tagging and pushing fcquic_server
The push refers to repository [docker.io/corentindetry/fcquic_server]

d544bb48: Wai

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

In [ ]:
import os
from grid5000 import Grid5000

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")
gk = Grid5000.from_yaml(conf_file)

print("Sites: ")
display(gk.sites.list())

Sites: 


[<Site uid:bordeaux>,
 <Site uid:grenoble>,
 <Site uid:lille>,
 <Site uid:louvain>,
 <Site uid:luxembourg>,
 <Site uid:lyon>,
 <Site uid:nancy>,
 <Site uid:nantes>,
 <Site uid:rennes>,
 <Site uid:sophia>,
 <Site uid:strasbourg>,
 <Site uid:toulouse>]

### Job configuration

Setup the various parameters for the job:
- The job name will identify the current booking, if the notebook kernel dies, re running the same reservation code with the same name will reload the existing job instead of booking a new one
- The walltime is the time that the booking will last, you can always stop your reservation earlier than the booking's end time

In [ ]:
JOB_NAME="fcquic_npf_test"
JOB_WALLTIME="0:20:00"
JOB_CLUSTER="spirou"
NUM_SERVER_VMS=1
NUM_CLIENT_VMS=2

### Create the booking

This first case will only create the booking object locally, but not yet place the booking.

You can adjust the different roles of the machines to match the roles defined in the NPF script.

For example, here I have 2 roles: server and client. The server node will run the server section of the NPF script,...

In [ ]:
import enoslib as en
from npf import enoslib as npf
from npf.output.transform.pandas import to_pandas
import logging

# Display some general information about the library
en.check()
# Enable rich logging
_ = en.init_logging()

from importlib import reload
reload(npf)

# conf = en.G5kConf.from_settings(job_name=JOB_NAME, walltime=JOB_WALLTIME).add_machine(
#     roles=["client"], cluster=JOB_CLUSTER, nodes=NUM_CLIENT_NODES
# ).add_machine(
#     roles=["server"], cluster=JOB_CLUSTER, nodes=NUM_SERVER_NODES
# )  
conf = (
    en.G5kConf.from_settings(job_name=JOB_NAME, walltime=JOB_WALLTIME)
    # For convenience, we use the site name as role
    .add_machine(roles=["server"], cluster="spirou", nodes=1)
    .add_machine(roles=["client"], cluster="gros", nodes=1)
)

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

[WARNING]: failed to patch stdout/stderr for fork-safety: 'OutStream' object
has no attribute 'buffer'
[WARNING]: failed to reconfigure stdout/stderr with custom encoding error
handler: 'OutStream' object has no attribute 'reconfigure'


In [4]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

Reserving resources...


Output()

Finished 1 tasks (Granting root access on the nodes (sudo-g5k)) on 
{'gros-66.nancy.grid5000.fr', 'spirou-5.louvain.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

{'server': {Host(address='spirou-5.louvain.grid5000.fr', alias='spirou-5.louvain.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'}, net_devices=set(), _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}, 'client': {Host(address='gros-66.nancy.grid5000.fr', alias='gros-66.nancy.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'}, net_devices=set(), _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}}

[G5k] gateway is not yet implemented for <class 'enoslib.infra.enos_g5k.objects.G5kEnosProd6Network'> on the G5k side
[G5k] gateway is not yet implemented for <class 'enoslib.infra.enos_g5k.objects.G5kEnosProd6Network'> on the G5k side


{'prod': {<enoslib.infra.enos_g5k.objects.G5kEnosProd4Network object at 0x738a992592e0>, <enoslib.infra.enos_g5k.objects.G5kEnosProd4Network object at 0x738a993dd520>, <enoslib.infra.enos_g5k.objects.G5kEnosProd6Network object at 0x738a9911d370>, <enoslib.infra.enos_g5k.objects.G5kEnosProd6Network object at 0x738a991babd0>}}

In [11]:
# From: https://discovery.gitlabpages.inria.fr/enoslib/tutorials/grid5000.html#kavlan-on-secondary-interfaces
# Fill in network information from nodes
roles = en.sync_info(roles, networks)

Output()

Finished 1 tasks (Waiting for connection) on {'spirou-5.louvain.grid5000.fr', 
'gros-66.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Finished 7 tasks (Gathering Facts,setup,utils : include_tasks,utils : Dump network 
information in a file,utils : Create the fake interfaces) on {'spirou-5.louvain.grid5000.fr',
'gros-66.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [10]:
display(roles)

ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
fe80::eaeb:d3ff:fefd:cf84/64 # noqa
172.16.208.5/20 # noqa
ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
fe80::9a03:9bff:feb0:c61e/64 # noqa


In [5]:
data = {}
# for node_role, hosts_data in roles.items():
#     for host_data in hosts_data:

#         # display(node_role)
#         # display(host_data)
#         # Get each node's IP address on the private network
#         ip_address_obj = host_data.filter_addresses(networks=networks["private"])[0]
#         # This may seem weird: ip_address_obj.ip is a `netaddr.IPv4Interface`
#         # which itself has an `ip` attribute.
#         node_ip = ip_address_obj.ip.ip
#         if data.get(node_role) is None:
#            data[node_role] = []
#         data[node_role].append(node_ip.exploded)

#         # display(node_ip)

# display(data)
# print(f"Server IP: {data["server"]}")
data["server"] = ["172.16.208.5"]

### Docker installation
Nodes don't have docker installed by default. Run `g5k-setup-docker -t` on every node to do so.

Could also use enoslib's [docker service](https://discovery.gitlabpages.inria.fr/enoslib/apidoc/docker.html#docker-service)

In [7]:
# runs on all nodes with the given roles and installs docker

with en.play_on(roles=roles) as p:
    p.shell('g5k-setup-docker -t')  

Output()

Finished 1 tasks (shell) on {'spirou-5.louvain.grid5000.fr', 'gros-66.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# TODO: use https://github.com/Emilevillette/enoslib-fastclick-test/blob/16d1155aa11ce1c5025bdce24476d3f3388cbb92/energy-scheduling/run-npf.py#L234
# to be able to set each client's IP directly from python here, and not make the clients fetch their local ip themselves

### Running the experiment

In [ ]:
# run the experiment...
print("Launching NPF")
import datetime
now = str(datetime.datetime.now())
results, _ = npf.run(
    "test_quic.npf",
    # Don't use series, it kinda breaks the whole test
    # series=[
    #     f"local",
    # ], 
    argsv= [
        "--single-output",
        f"./npf-out/{now}.csv",
        "--force-retest",
        "--no-graph",
        f"--variables",
            f"SERVER_IP={data["server"][0]}", 
    ],
    roles=roles,
)
df = to_pandas(results)

display(df)

Launching NPF
cluster/localhost.node could not be found, we will connect to localhost with SSH using default parameters
cluster/spirou-5.louvain.grid5000.fr.node could not be found, we will connect to spirou-5.louvain.grid5000.fr with SSH using default parameters
cluster/gros-66.nancy.grid5000.fr.node could not be found, we will connect to gros-66.nancy.grid5000.fr with SSH using default parameters
[Local] Running test test_quic.npf...
FCQUIC vs QUIC vs TCP (with TLS)


Executing init scripts...


Output()

Output()

Finished 1 tasks ( bash -c 'mkdir -p test2603160904-05232 && cd test2603160904-05232;
# first use sudo-g5k to be able to use sudo for the rest of the experiment
sudo-g5k
# Setting buffer values
sudo sysctl -w net.core.rmem_max=26214400
sudo sysctl -w net.core.rmem_max=26214400



') on {'gros-66.nancy.grid5000.fr'}

...

─────────────────────────────────────────────────────────────────────────────────────────────

server [0] You already have sudo permissions.
server [0] net.core.rmem_max = 26214400
server [0] net.core.rmem_max = 26214400


Output()

INTERVAL = 100, LAMBDA = 0.1, NUM_CLIENTS = 1, MIN = 1, TEST_LENGTH = 10, FAL = 400, ADD = 0, POISSON = "true", TCP_NODELAY = "true", USE = "false", PER = false, WORKDIR = "/home/cdetry", SERVER_IP = 172.16.208.5, CLIENT_IPS = "" [run 1/3 for test 1/3]


Output()

In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(f"./npf-out/{now}.csv")

# 2. Lineplot: latency vs additional data size
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(
    data=df,
    x="ADDITIONAL_DATA_SIZE",
    y="y_LATENCY",
    markers=True,
    errorbar="sd",
    ax=ax,
)
ax.set_xlabel("Additional Data Size")
ax.set_ylabel("Latency (ms)")
ax.set_title("Latency vs Additional Data Size")
plt.tight_layout()
plt.savefig("latency_vs_size.png", dpi=150)
plt.show()

plt.figure()
# 4. CDF of latency
fig, ax = plt.subplots(figsize=(8, 5))
sns.ecdfplot(data=df, x="y_LATENCY", ax=ax)
ax.set_xlabel("Latency (ms)")
ax.set_title("CDF of Latency")
plt.tight_layout()
plt.savefig("latency_cdf.png", dpi=150)
plt.show()

### Stopping the current booking

In [9]:
provider.destroy()